### Data Ingestion

In [1]:
###Document Structure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="this is the main text content. I am using it to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Piyush",
        "date_created":"2026-04-04"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Piyush', 'date_created': '2026-04-04'}, page_content='this is the main text content. I am using it to create RAG')

In [3]:
## create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [5]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

C:\Users\pshma\AppData\Local\Temp\ipykernel_8980\2087070391.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\RAG Pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

### Load all the text file from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",  ## Pattern to match the file
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents = dir_loader.load()
documents 

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [7]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

### Load all the text file from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",  ## Pattern to match the file
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents


[Document(metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'TeX', 'creationdate': '2022-09-05T11:27:55+00:00', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 14, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-09-05T11:27:55+00:00', 'trapped': '', 'modDate': 'D:20220905112755Z', 'creationDate': 'D:20220905112755Z', 'page': 0}, page_content='BSCCS2005: Extra Coding Sessions\nPractice Questions\n1. Write Java code as instructed.\n[Basic class and method]\n• Define class Employee that has the following members:\n– Private instance variables String empName, double salary\n– Mutator methods to update the instance variables\n– Accessor methods to access the instance variables\n– Method public double bonus(float percent) that returns the bonus com-\nputed as (percent/100.0)*salary\n• Define class EmpTest that has the main method and the following functionalities:\n– Create an Employee obje

### Creating Data Chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    split document into smaller chunks for better RAG perfomance

    ARG:
        chunk_size: Maximum character per chunk (adjust based on your LLM)
        chunk_overlap: Characters to overlap between chunks (preserve context)
    """
    text_splitters = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitters.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Context: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [9]:
chunks=split_documents(pdf_documents)
chunks

Split 26 documents into 49 chunks

Example chunk:
Context: BSCCS2005: Extra Coding Sessions
Practice Questions
1. Write Java code as instructed.
[Basic class and method]
• Define class Employee that has the following members:
– Private instance variables Stri...
Metadata: {'producer': 'pdfTeX-1.40.23', 'creator': 'TeX', 'creationdate': '2022-09-05T11:27:55+00:00', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 14, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-09-05T11:27:55+00:00', 'trapped': '', 'modDate': 'D:20220905112755Z', 'creationDate': 'D:20220905112755Z', 'page': 0}


[Document(metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'TeX', 'creationdate': '2022-09-05T11:27:55+00:00', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 14, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-09-05T11:27:55+00:00', 'trapped': '', 'modDate': 'D:20220905112755Z', 'creationDate': 'D:20220905112755Z', 'page': 0}, page_content='BSCCS2005: Extra Coding Sessions\nPractice Questions\n1. Write Java code as instructed.\n[Basic class and method]\n• Define class Employee that has the following members:\n– Private instance variables String empName, double salary\n– Mutator methods to update the instance variables\n– Accessor methods to access the instance variables\n– Method public double bonus(float percent) that returns the bonus com-\nputed as (percent/100.0)*salary\n• Define class EmpTest that has the main method and the following functionalities:\n– Create an Employee obje

### Embedding and vectorStoreDB

In [10]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    """Handels document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTrnasformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embedding for a list of texts

            Args:
                texts: List of text strings to embed

            Returns:
                numpy array of embedding with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager
        

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6769.88it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [12]:
class VectorStore:
    """Manages documents embedding in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            presist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB and collection"""
        try:
            # Create persistance ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embedding for RAG",
                    "hnsw:space": "cosine" # <--- ADD THIS LINE
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

            

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [13]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'TeX', 'creationdate': '2022-09-05T11:27:55+00:00', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 14, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-09-05T11:27:55+00:00', 'trapped': '', 'modDate': 'D:20220905112755Z', 'creationDate': 'D:20220905112755Z', 'page': 0}, page_content='BSCCS2005: Extra Coding Sessions\nPractice Questions\n1. Write Java code as instructed.\n[Basic class and method]\n• Define class Employee that has the following members:\n– Private instance variables String empName, double salary\n– Mutator methods to update the instance variables\n– Accessor methods to access the instance variables\n– Method public double bonus(float percent) that returns the bonus com-\nputed as (percent/100.0)*salary\n• Define class EmpTest that has the main method and the following functionalities:\n– Create an Employee obje

In [14]:
## lets convert the chunks to embedding
texts=[doc.page_content for doc in chunks]

## Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# Store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embedding for 49 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.23s/it]

Generated embeddings with shape: (49, 384)
Adding 49 documents to vector store...
Successfully added 49 documents to vector store
Total documents in collection: 49


Retriver Pipeline from VectorStore

In [15]:
class RAGRetriver:
    """Handles query-based retrival from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List fo dictionary containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score(ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrived {len(retrieved_docs)} documnets (after filtering)")
                
                # Use a proper if statement
                if not retrieved_docs:
                    print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever=RAGRetriver(vectorstore, embedding_manager)


In [16]:
rag_retriever

In [ ]:
rag_retriever.retrieve("what is interface printable")

In [ ]:
rag_retriever.retrieve("what is const cancelAppointment")

Integration Vectordb Context pipeline With LLM output

In [18]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in enviroment)
groq_api_key = os.environ.get("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key, 
    model_name="llama-3.1-8b-instant", 
    temperature=0.1, 
    max_tokens=1024
)

## 2. Simple RAG function: retrieve context + genarate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "NO relevant context found to the answer the question."

    ## generate the answer using GROQ LLM
    prompt="""Use the following context to answer the question concisely:
        Context:
        {context}

        Question: {query}

        Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [20]:
answer=rag_simple("how the cancelAppointment function is working and whats the code for it?", rag_retriever,llm)
print(answer)

Retrieving documents for query: 'how the cancelAppointment function is working and whats the code for it?'
Top K: 3, Score threshold: 0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.33it/s]

Generated embeddings with shape: (1, 384)
Retrived 3 documnets (after filtering)


Based on the provided context, there is no code for the `cancelAppointment` function. However, I can provide a possible implementation of the `cancelAppointment` function based on the given context.

```javascript
function cancelAppointment() {
    try {
        // Assuming there's a fetch call to cancel the appointment
        fetch('/cancel-appointment', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json'
            }
        })
        .then(response => response.json())
        .then(data => {
            // If the history tab is open, refresh it to show the cancelled status
            if (activeTab.value === 'history') {
                fetchHistory();
            }
        })
        .catch(error => {
            console.error("Failed to cancel appointment:", error);
            alert("Failed to cancel appointment. Please try again.");
        });
    } catch (error) {
        console.error("Failed to cancel appointment:", erro

In [21]:
# Fixed spelling and increased top_k to 5 to give the LLM more reading material
answer = rag_simple(
    query="how the cancelAppointment function is working and whats the code for it?", 
    retriever=rag_retriever,
    llm=llm,
    top_k=5 
)
print(answer)

Retrieving documents for query: 'how the cancelAppointment function is working and whats the code for it?'
Top K: 5, Score threshold: 0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.61it/s]

Generated embeddings with shape: (1, 384)
Retrived 5 documnets (after filtering)


Based on the provided context, it appears that the `cancelAppointment` function is not explicitly defined in the given code snippets. However, based on the surrounding code, it seems to be related to canceling an appointment and refreshing the dashboard.

Here's a possible implementation of the `cancelAppointment` function:

```javascript
function cancelAppointment() {
    try {
        // Code to cancel the appointment goes here
        // For example:
        fetchDashboard();
        
        // If they have the history tab open, refresh that too so it shows as "Cancelled"
        if (activeTab.value === 'history') {
            fetchHistory();
        }
    } catch (error) {
        console.error("Failed to cancel appointment:", error);
        alert("Failed to cancel appointment. Please try again.");
    }
}
```

This function attempts to cancel the appointment, refreshes the dashboard, and if the history tab is open, it also refreshes that tab to show the appointment as "Cancelle